# COMP 469 Lab 02: Solving Problems by Searching

**Guided student variant** | **Week 2** | **AIMA Chapter 3** | **3-hour lab**

## Lab Goal

In this lab, you will formulate a route-finding problem and implement the core search algorithms from AIMA Chapter 3:

- Uninformed search: breadth-first graph search and depth-first graph search.
- Cost-aware search: uniform-cost search.
- Informed search: greedy best-first search and A* search.

You will compare these algorithms on the Romania road map, then extend one heuristic to the 8-puzzle. Along the way you will run a controlled experiment measuring nodes expanded, path cost, and running time for each algorithm, and plot the results.

This guided version gives you structure inside the code TODOs. Your job is still to complete each TODO, answer the checkpoint questions in your own words, and finish the reflection at the end.

**Estimated pacing (3 hours):**

| Section | Time |
|---|---|
| 0-2. Setup and problem formulation review | 25 min |
| 3-4. Define and visualize the Romania problem | 15 min |
| 5-6. Uninformed search: BFS and DFS graph search | 35 min |
| 7. Uniform-cost search | 20 min |
| 8-9. Heuristics, greedy best-first search, and A* | 35 min |
| 10. 8-Puzzle heuristics | 20 min |
| 11-12. Controlled experiment and plots | 30 min |
| 13. Extension challenge (optional) | 15 min |
| 14. Final reflection | 5 min |


## 0. Python `.env` Setup

If you already completed this setup for Lab 01, activate that same environment and skip to Section 1.

### First-Time Setup

Run these commands from the folder where this notebook is located.

**macOS/Linux:**

```bash
python3 -m venv .env
source .env/bin/activate
python -m pip install --upgrade pip
python -m pip install ipykernel notebook jupyter matplotlib networkx ipywidgets
python -m ipykernel install --user --name comp469 --display-name "COMP 469 (.env)"
```

**Windows PowerShell:**

```powershell
py -m venv .env
.\.env\Scripts\Activate.ps1
python -m pip install --upgrade pip
python -m pip install ipykernel notebook jupyter matplotlib networkx ipywidgets
python -m ipykernel install --user --name comp469 --display-name "COMP 469 (.env)"
```

If PowerShell blocks activation, run this once in the same terminal:

```powershell
Set-ExecutionPolicy -ExecutionPolicy RemoteSigned -Scope Process
.\.env\Scripts\Activate.ps1
```

### Returning Later

Always activate the environment before launching Jupyter.

**macOS/Linux:**

```bash
source .env/bin/activate
jupyter notebook
```

**Windows PowerShell:**

```powershell
.\.env\Scripts\Activate.ps1
jupyter notebook
```

In Jupyter, choose the kernel named **COMP 469 (.env)** if it appears.

### aima/ Support Package

This notebook depends on the bundled `aima/` package (the `search.py` and `notebook_utils.py` modules from the AIMA code repository). Make sure the `aima/` folder is in this notebook's directory, or in a parent directory, before you run the setup cell below.


## 1. Setup Cell

Run this cell first. Do not edit it unless your instructor asks you to.


In [ ]:
import os, sys

# Make the bundled AIMA support package importable whether the notebook
# is launched from this folder or from the repository root.
_root = os.path.abspath(os.getcwd())
while _root != os.path.dirname(_root) and not os.path.isdir(os.path.join(_root, 'aima')):
    _root = os.path.dirname(_root)

if not os.path.isdir(os.path.join(_root, 'aima')):
    raise FileNotFoundError('Could not find the bundled aima/ support package')

if _root not in sys.path:
    sys.path.insert(0, _root)

import random
import time
from collections import deque

import matplotlib.pyplot as plt
import networkx as nx

from aima.search import Problem, Node, GraphProblem, UndirectedGraph, memoize, PriorityQueue, infinity
from aima.notebook_utils import psource, show_map

import warnings
warnings.filterwarnings("ignore")

random.seed(469)
print("Setup complete. You are ready for Lab 02.")


## 2. Problem Formulation Review

AIMA Chapter 3 defines a search problem with five parts: a **state space**, an **initial state**, an **actions**/**transition model** pair, a **goal test**, and an **action cost** function. A solution is a path from the initial state to a goal state, and an optimal solution has the lowest path cost among all solutions.

### PEAS-style problem formulation

Complete the table for the route-finding problem described in Chapter 3 (an agent in Arad must reach Bucharest).

| Component | Your Answer |
|---|---|
| State space | TODO |
| Initial state | TODO |
| Goal test | TODO |
| Actions | TODO |
| Action cost | TODO |

Now classify the environment using AIMA vocabulary.

| Property | Choice | Brief justification |
|---|---|---|
| Fully or partially observable? | TODO | TODO |
| Single-agent or multi-agent? | TODO | TODO |
| Deterministic or stochastic? | TODO | TODO |
| Episodic or sequential? | TODO | TODO |
| Static or dynamic? | TODO | TODO |
| Discrete or continuous? | TODO | TODO |

### The `Problem` and `Node` classes

Run the next two cells to see how AIMA defines the abstract `Problem` class and the `Node` class that `search.py` uses to build the search tree.


In [ ]:
psource(Problem)


**Short answer (2-3 sentences):** `Problem` exposes a `goal_test(state)` method rather than a single fixed goal state. Why is this more general, and what kind of problem would need it (give one example besides route-finding)?

TODO: your answer here.


In [ ]:
psource(Node)


**Short answer (2-3 sentences):** A `Node` stores `state`, `parent`, `action`, and `path_cost`, but the state space graph itself does not store any of these. Why does the search tree need this extra bookkeeping that the state space graph does not?

TODO: your answer here.


## 3. Define the Romania Map and the Route-Finding Problem

`romania_map` is an `UndirectedGraph`: a dictionary of nodes as keys and `{neighbor: distance}` dictionaries as values. This is the same map shown in Figure 3.1 of AIMA.


In [ ]:
romania_map = UndirectedGraph(dict(
    Arad=dict(Zerind=75, Sibiu=140, Timisoara=118),
    Bucharest=dict(Urziceni=85, Pitesti=101, Giurgiu=90, Fagaras=211),
    Craiova=dict(Drobeta=120, Rimnicu=146, Pitesti=138),
    Drobeta=dict(Mehadia=75),
    Eforie=dict(Hirsova=86),
    Fagaras=dict(Sibiu=99),
    Hirsova=dict(Urziceni=98),
    Iasi=dict(Vaslui=92, Neamt=87),
    Lugoj=dict(Timisoara=111, Mehadia=70),
    Oradea=dict(Zerind=71, Sibiu=151),
    Pitesti=dict(Rimnicu=97),
    Rimnicu=dict(Sibiu=80),
    Urziceni=dict(Vaslui=142),
))
romania_map.locations = dict(
    Arad=(91, 492), Bucharest=(400, 327), Craiova=(253, 288), Drobeta=(165, 299),
    Eforie=(562, 293), Fagaras=(305, 449), Giurgiu=(375, 270), Hirsova=(534, 350),
    Iasi=(473, 506), Lugoj=(165, 379), Mehadia=(168, 339), Neamt=(406, 537),
    Oradea=(131, 571), Pitesti=(320, 368), Rimnicu=(233, 410), Sibiu=(207, 457),
    Timisoara=(94, 410), Urziceni=(456, 350), Vaslui=(509, 444), Zerind=(108, 531),
)

# TODO 1: Create the route-finding problem described in Section 3.1: the agent starts
# in Arad and must reach Bucharest.
# Guided hint: GraphProblem(initial_state, goal_state, graph)
romania_problem = GraphProblem("TODO", "TODO", romania_map)

print("Initial state:", romania_problem.initial)
print("Is Bucharest the goal?", romania_problem.goal_test("Bucharest"))


## 4. Visualize the Map

We use `networkx` to lay out the map and `show_map` (bundled in `aima/notebook_utils.py`) to draw it, matching Figure 3.1 in the book.


In [ ]:
node_colors = {node: 'white' for node in romania_map.locations.keys()}
node_positions = romania_map.locations
node_label_pos = {k: [v[0], v[1] - 10] for k, v in romania_map.locations.items()}
edge_weights = {(k, k2): v2 for k, v in romania_map.graph_dict.items() for k2, v2 in v.items()}

romania_graph_data = {
    'graph_dict': romania_map.graph_dict,
    'node_colors': node_colors,
    'node_positions': node_positions,
    'node_label_positions': node_label_pos,
    'edge_weights': edge_weights,
}

show_map(romania_graph_data)


### Checkpoint 1

Answer in 4-6 sentences:

1. What is the state space for this route-finding problem, and how many states does it contain?
2. What do the edge weights in `romania_map` represent, and how are they used by the action cost function?
3. Is `romania_map` best explored with tree search or graph search? Explain using a specific pair of cities that have more than one path between them.
4. `romania_map.locations` stores an (x, y) coordinate for every city, but only the distances in `romania_map.graph_dict` are needed to compute path cost. What is the coordinate data used for instead?


## 5. Breadth-First Graph Search (Uninformed)

Breadth-first search explores the search tree level by level: it expands all nodes at depth *d* before any node at depth *d+1*. This guarantees the **shallowest** goal node is found first, which means BFS is optimal when every action has the same cost, but it is not guaranteed to find the lowest-cost path when action costs differ (as they do here, since road distances vary).

Your task:

- Use a FIFO queue (`collections.deque`) as the frontier.
- Track an `explored` set so no state is expanded twice.
- Count every node you pop from the frontier and expand as one "expansion" -- you will use this count later to compare algorithms.


In [ ]:
def breadth_first_graph_search(problem):
    """Search the shallowest nodes in the search tree first. Returns (goal_node, nodes_expanded)
    or (None, nodes_expanded) if no solution exists."""
    nodes_expanded = 0

    node = Node(problem.initial)
    if problem.goal_test(node.state):
        return node, nodes_expanded

    frontier = deque([node])
    explored = set()

    while frontier:
        # TODO 2: Pop the next node from the LEFT of the frontier (FIFO order).
        node = None  # TODO: replace with the correct deque operation.

        explored.add(node.state)
        nodes_expanded += 1

        for child in node.expand(problem):
            # TODO 3: A child should only be considered if its state is not already
            # explored AND not already sitting in the frontier.
            # Guided hint: "child not in frontier" works because Node defines __eq__ by state.
            if True:  # TODO: replace True with the correct condition.
                if problem.goal_test(child.state):
                    return child, nodes_expanded
                frontier.append(child)

    return None, nodes_expanded


bfs_solution, bfs_expanded = breadth_first_graph_search(romania_problem)
print("Path:", bfs_solution.solution())
print("Path cost:", bfs_solution.path_cost)
print("Nodes expanded:", bfs_expanded)


## 6. Depth-First Graph Search (Uninformed)

Depth-first search always expands the **deepest** unexpanded node first. It uses a LIFO stack instead of a FIFO queue -- that is the only structural difference from breadth-first search in this implementation.

Your task:

- Reuse the same overall structure as `breadth_first_graph_search`.
- Use a plain Python `list` as a stack (`frontier.append(...)` / `frontier.pop()`).


In [ ]:
def depth_first_graph_search(problem):
    """Search the deepest nodes in the search tree first. Returns (goal_node, nodes_expanded)
    or (None, nodes_expanded) if no solution exists."""
    nodes_expanded = 0

    node = Node(problem.initial)
    if problem.goal_test(node.state):
        return node, nodes_expanded

    frontier = [node]
    explored = set()

    while frontier:
        # TODO 4: Pop the next node from the frontier so that it behaves like a stack (LIFO).
        node = None  # TODO: replace with the correct list operation.

        explored.add(node.state)
        nodes_expanded += 1

        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                if problem.goal_test(child.state):
                    return child, nodes_expanded
                # TODO 5: Add the child to the frontier.
                pass

    return None, nodes_expanded


dfs_solution, dfs_expanded = depth_first_graph_search(romania_problem)
print("Path:", dfs_solution.solution())
print("Path cost:", dfs_solution.path_cost)
print("Nodes expanded:", dfs_expanded)


### Checkpoint 2

Answer in 4-6 sentences:

1. Compare the BFS and DFS paths from Arad to Bucharest. Are they the same path? Do they have the same path cost?
2. Which algorithm expanded more nodes on this problem? Was that what you expected, and why?
3. Neither of these algorithms uses `romania_map`'s edge weights when deciding what to expand next. What does that tell you about whether BFS or DFS is guaranteed to find the optimal (lowest-cost) path?
4. Under what circumstances (if any) would breadth-first search be guaranteed to return the optimal path?


## 7. Uniform-Cost Search (Cost-Aware)

Uniform-cost search always expands the node in the frontier with the lowest **path cost** `g(n)` so far. Unlike BFS/DFS, it is guaranteed to find the optimal solution whenever action costs are non-negative, because it never expands a node before all cheaper paths have been considered.

We use AIMA's own `PriorityQueue('min', f)` so the frontier automatically pops the lowest-cost node, and `memoize` so `f` is not recomputed for the same node.

Your task:

- Set `f` to be the path cost of a node (`node.path_cost`).
- Handle the case where a child reaches a state already in the frontier with a *higher* cost: replace it.


In [ ]:
def uniform_cost_search(problem):
    """Best-first graph search with f(n) = g(n) = path cost. Returns (goal_node, nodes_expanded)
    or (None, nodes_expanded) if no solution exists."""
    nodes_expanded = 0

    # TODO 6: Define f as the path cost of a node. Guided hint: a lambda taking one argument, n.
    f = memoize(lambda n: None, 'f')  # TODO: replace None with the path-cost expression.

    node = Node(problem.initial)
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    explored = set()

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            return node, nodes_expanded

        explored.add(node.state)
        nodes_expanded += 1

        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                frontier.append(child)
            elif child in frontier:
                incumbent = frontier[child]
                # TODO 7: If the newly found path to child is cheaper than the incumbent
                # path already in the frontier, remove the incumbent and append the
                # cheaper child instead. Guided hint: compare f(child) to incumbent.
                pass

    return None, nodes_expanded


ucs_solution, ucs_expanded = uniform_cost_search(romania_problem)
print("Path:", ucs_solution.solution())
print("Path cost:", ucs_solution.path_cost)
print("Nodes expanded:", ucs_expanded)


## 8. A Heuristic: Straight-Line Distance

Informed search algorithms use a heuristic function `h(n)` that estimates the cost from a node's state to the nearest goal. For route-finding, the classic heuristic is the **straight-line distance** between two cities, using the (x, y) coordinates in `romania_map.locations`.

A heuristic is **admissible** if it never overestimates the true remaining cost. Straight-line distance is admissible for road distance because a straight line is always the shortest path between two points -- any real road can only be the same length or longer.

Your task:

- Compute the Euclidean distance between the (x, y) coordinates of `state` and `goal`.


In [ ]:
import math

def straight_line_distance(state, goal, locations):
    x1, y1 = locations[state]
    x2, y2 = locations[goal]

    # TODO 8: Compute the Euclidean distance between (x1, y1) and (x2, y2).
    return 0.0  # TODO: replace with the correct formula.


def romania_h(node):
    return straight_line_distance(node.state, romania_problem.goal, romania_map.locations)


# Quick sanity check: Bucharest to itself should have heuristic 0.
print("h(Bucharest) =", straight_line_distance("Bucharest", "Bucharest", romania_map.locations))
print("h(Arad) =", romania_h(Node("Arad")))


## 9. Greedy Best-First Search and A* Search (Informed)

Both algorithms are **best-first searches**: they always expand the node in the frontier with the lowest `f(n)`. They differ only in how `f` is defined:

- **Greedy best-first search**: `f(n) = h(n)`. Fast, but can be led down expensive paths because it ignores cost already paid.
- **A\* search**: `f(n) = g(n) + h(n)`. Optimal when `h` is admissible, because it balances cost-so-far against estimated cost-to-go.

Complete the shared `best_first_graph_search` helper once, then define greedy and A\* by passing in different `f` functions.


In [ ]:
def best_first_graph_search(problem, f):
    """Search nodes with the lowest f(n) first. Returns (goal_node, nodes_expanded)
    or (None, nodes_expanded) if no solution exists."""
    nodes_expanded = 0
    f = memoize(f, 'f')

    node = Node(problem.initial)
    frontier = PriorityQueue('min', f)
    frontier.append(node)
    explored = set()

    while frontier:
        node = frontier.pop()

        if problem.goal_test(node.state):
            return node, nodes_expanded

        explored.add(node.state)
        nodes_expanded += 1

        for child in node.expand(problem):
            if child.state not in explored and child not in frontier:
                frontier.append(child)
            elif child in frontier:
                incumbent = frontier[child]
                if f(child) < incumbent:
                    del frontier[child]
                    frontier.append(child)

    return None, nodes_expanded


def greedy_best_first_search(problem, h):
    # TODO 9: Set f(n) = h(n). Guided hint: lambda n: h(n)
    return best_first_graph_search(problem, lambda n: None)  # TODO: replace None.


def astar_search(problem, h):
    # TODO 10: Set f(n) = g(n) + h(n). Guided hint: n.path_cost is g(n).
    return best_first_graph_search(problem, lambda n: None)  # TODO: replace None.


greedy_solution, greedy_expanded = greedy_best_first_search(romania_problem, romania_h)
print("Greedy path:", greedy_solution.solution())
print("Greedy path cost:", greedy_solution.path_cost)
print("Greedy nodes expanded:", greedy_expanded)
print()

astar_solution, astar_expanded = astar_search(romania_problem, romania_h)
print("A* path:", astar_solution.solution())
print("A* path cost:", astar_solution.path_cost)
print("A* nodes expanded:", astar_expanded)


### Checkpoint 3

Answer in 4-6 sentences:

1. Compare the greedy best-first path and the A* path. Do they reach the same path cost as uniform-cost search? If not, explain why greedy search might return a more expensive path.
2. Compare nodes expanded across UCS, greedy, and A* on Arad to Bucharest. Which expanded the fewest? Which expanded the most?
3. `straight_line_distance` is admissible. What would likely happen to A*'s optimality guarantee if you used a heuristic that could overestimate the true distance (for example, straight-line distance multiplied by 2)?
4. Is greedy best-first search complete (guaranteed to find a solution if one exists) on a finite graph like this one? Is it optimal? Explain the difference between these two properties.


## 10. The 8-Puzzle and Heuristic Design

The 8-puzzle (Section 3.2.1) is a sliding-tile puzzle: a 3x3 grid with eight numbered tiles and one blank space. States are represented as a tuple of 9 values (0 represents the blank), read left-to-right, top-to-bottom.

Two classic heuristics for the 8-puzzle:

1. **Misplaced tiles**: the number of tiles that are not in their goal position.
2. **Manhattan distance**: the sum, over every tile, of the number of rows plus columns it is away from its goal position.

Both are admissible for the same reason straight-line distance is admissible for route-finding: each is a relaxation of the true cost (every action moves exactly one tile by one square), so neither heuristic can overestimate the number of moves actually required.

Your task: complete both heuristic functions below. Each takes a `Node` whose `.state` is a length-9 tuple.


In [ ]:
goal_state = (1, 2, 3, 4, 5, 6, 7, 8, 0)

# index_goal[tile] = (row, col) of that tile in the goal state
index_goal = {1: (0, 0), 2: (0, 1), 3: (0, 2),
              4: (1, 0), 5: (1, 1), 6: (1, 2),
              7: (2, 0), 8: (2, 1), 0: (2, 2)}


def misplaced_tiles(node):
    state = node.state
    # TODO 11: Count how many positions in `state` do not match `goal_state`.
    # Guided hint: the blank tile (0) is usually excluded from this count, but including
    # it does not break admissibility here -- decide which you prefer and be ready to
    # justify it in Checkpoint 4.
    count = 0
    for i in range(9):
        pass  # TODO: increment count when state[i] != goal_state[i] (and, optionally, state[i] != 0).
    return count


def manhattan_distance(node):
    state = node.state
    total = 0
    for i in range(9):
        tile = state[i]
        if tile == 0:
            continue
        current_row, current_col = i // 3, i % 3
        goal_row, goal_col = index_goal[tile]
        # TODO 12: Add the Manhattan distance for this tile (|row difference| + |col difference|)
        # to `total`.
        pass
    return total


test_state = (2, 4, 3, 1, 5, 6, 7, 8, 0)
test_node = Node(test_state)
print("Misplaced tiles:", misplaced_tiles(test_node))
print("Manhattan distance:", manhattan_distance(test_node))


In [ ]:
class EightPuzzleProblem(Problem):
    """A minimal 8-puzzle Problem, independent of any pre-built AIMA puzzle class."""

    def __init__(self, initial, goal=goal_state):
        super().__init__(initial, goal)

    def actions(self, state):
        blank = state.index(0)
        row, col = blank // 3, blank % 3
        possible = []
        if row > 0:
            possible.append("Up")
        if row < 2:
            possible.append("Down")
        if col > 0:
            possible.append("Left")
        if col < 2:
            possible.append("Right")
        return possible

    def result(self, state, action):
        blank = state.index(0)
        row, col = blank // 3, blank % 3
        delta = {"Up": -3, "Down": 3, "Left": -1, "Right": 1}
        swap_with = blank + delta[action]
        state = list(state)
        state[blank], state[swap_with] = state[swap_with], state[blank]
        return tuple(state)

    def goal_test(self, state):
        return state == self.goal

    def path_cost(self, c, state1, action, state2):
        return c + 1


puzzle = EightPuzzleProblem((2, 4, 3, 1, 5, 6, 7, 8, 0))

for name, h in [("misplaced_tiles", misplaced_tiles), ("manhattan_distance", manhattan_distance)]:
    start = time.perf_counter()
    solution, expanded = astar_search(puzzle, h)
    elapsed = time.perf_counter() - start
    print(f"{name}: moves={len(solution.solution())}, nodes_expanded={expanded}, time={elapsed:.4f}s")


### Checkpoint 4

Answer in 4-6 sentences:

1. Both heuristics found a solution with the same number of moves. Why must that be true if both are admissible and A* is being used?
2. Which heuristic expanded fewer nodes? A heuristic that is closer to the true cost while still never overestimating it is called **more informed**. What does your result tell you about which heuristic is more informed?
3. If you included the blank tile in your `misplaced_tiles` count, would the heuristic still be admissible? Why or why not?
4. Manhattan distance sums an independent estimate per tile. Explain, in your own words, why this relaxation (ignoring that tiles block each other) can never overestimate the true number of moves.


## 11. Controlled Experiment

To compare algorithms fairly, each algorithm should be run against the same set of route-finding problems. The list below picks several (start, goal) pairs across the Romania map, including some short and some long routes.


In [ ]:
route_pairs = [
    ("Arad", "Bucharest"),
    ("Oradea", "Eforie"),
    ("Timisoara", "Vaslui"),
    ("Neamt", "Craiova"),
    ("Zerind", "Hirsova"),
]

algorithms = {
    "BFS": lambda problem: breadth_first_graph_search(problem),
    "DFS": lambda problem: depth_first_graph_search(problem),
    "UCS": lambda problem: uniform_cost_search(problem),
    "Greedy": lambda problem: greedy_best_first_search(
        problem, lambda n: straight_line_distance(n.state, problem.goal, romania_map.locations)
    ),
    "A*": lambda problem: astar_search(
        problem, lambda n: straight_line_distance(n.state, problem.goal, romania_map.locations)
    ),
}


def run_experiment(route_pairs, algorithms):
    rows = []

    for start, goal in route_pairs:
        problem = GraphProblem(start, goal, romania_map)

        for algo_name, run_algo in algorithms.items():
            # TODO 13: Time the algorithm call using time.perf_counter() before and after,
            # then compute elapsed = end - start.
            start_time = None  # TODO: record the time before running the algorithm.
            solution, nodes_expanded = run_algo(problem)
            end_time = None    # TODO: record the time after running the algorithm.
            elapsed = 0.0      # TODO: compute end_time - start_time.

            rows.append({
                "algorithm": algo_name,
                "start": start,
                "goal": goal,
                "path_cost": solution.path_cost if solution else None,
                "nodes_expanded": nodes_expanded,
                "time_seconds": elapsed,
            })

    return rows


results = run_experiment(route_pairs, algorithms)
results[:5]


## 12. Summarize and Plot Your Results

Create at least two plots:

1. Average nodes expanded by algorithm.
2. Average path cost by algorithm.

You may add more plots (for example, average running time) if they help your analysis.


In [ ]:
def summarize_results(results):
    summary = {}
    algo_names = sorted({row["algorithm"] for row in results})

    for algo_name in algo_names:
        rows = [row for row in results if row["algorithm"] == algo_name]
        summary[algo_name] = {
            "avg_nodes_expanded": sum(row["nodes_expanded"] for row in rows) / len(rows),
            "avg_path_cost": sum(row["path_cost"] for row in rows) / len(rows),
            "avg_time_seconds": sum(row["time_seconds"] for row in rows) / len(rows),
        }

    return summary


summary = summarize_results(results)
summary


In [ ]:
def plot_summary_metric(summary, metric, title, ylabel):
    names = list(summary.keys())
    values = [summary[name][metric] for name in names]

    plt.figure(figsize=(9, 4))
    plt.bar(names, values)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


plot_summary_metric(summary, "avg_nodes_expanded", "Average Nodes Expanded by Algorithm", "Average nodes expanded")
plot_summary_metric(summary, "avg_path_cost", "Average Path Cost by Algorithm", "Average path cost (km)")


### Checkpoint 5

Answer in 5-7 sentences:

1. Which algorithm had the lowest average path cost? Was this expected given what you know about each algorithm's guarantees?
2. Which algorithm expanded the fewest nodes on average? Did that algorithm also have the lowest path cost? If not, explain the tradeoff.
3. Did DFS ever find a path as cheap as A*'s on any of the five routes? What does that suggest about relying on DFS results without checking them?
4. How did the choice of heuristic (straight-line distance) affect A*'s node count compared to UCS, which uses no heuristic at all?
5. If you needed a routing system that had to run instantly on a device with very little memory, which algorithm from this lab would you pick, and what would you be giving up?


## 13. Extension Challenge

Choose one extension if you finish early.

### Option A: Iterative Deepening Search

Implement iterative deepening search: repeatedly run a depth-limited depth-first search with an increasing depth limit (0, 1, 2, ...) until a solution is found. Compare its nodes-expanded count against plain BFS and DFS on a route with a long path (e.g., Oradea to Eforie).

### Option B: Weighted A*

Modify `astar_search` so that `f(n) = g(n) + w * h(n)` for a weight `w` you can pass in. Try `w = 0` (this should behave like UCS), `w = 1` (plain A*), and `w = 3` (more greedy). For each weight, record path cost and nodes expanded, and discuss the tradeoff you observe.

### Option C: A New 8-Puzzle Heuristic

Design and implement a third heuristic for the 8-puzzle (for example, "max(misplaced_tiles, manhattan_distance)" or your own idea). Verify it is admissible by reasoning through a few states by hand, then compare its nodes-expanded count against `misplaced_tiles` and `manhattan_distance` on two or three puzzle instances.


In [ ]:
# Extension workspace.
# Put your optional extension code here.


## 14. Final Reflection

Write one well-developed paragraph answering the prompt below.

> Across the algorithms in this lab, what is the relationship between how much information a search algorithm uses (none, path cost only, or path cost plus a heuristic) and the quality of the solution it finds? Is more information always worth the extra bookkeeping cost? Use evidence from your experiment and vocabulary from AIMA Chapter 3 (completeness, optimality, admissibility, time and space complexity).

## Submission Checklist

Before submitting, confirm that:

- Your notebook runs from top to bottom without errors.
- All TODO sections are completed or clearly attempted.
- All five checkpoints are answered.
- Your experiment summary is visible.
- Your plots are visible.
- Your final reflection is complete.
